# Introduction: Ollama LLM Backend

Ollama is a powerful open-source platform for running large language models (LLMs) locally on your machine. It operates as an independent service with a built-in server that exposes an OpenAI-compatible API endpoint at http://localhost:11434/v1.

This makes Ollama perfect for AI agent development, local experimentation, and production deployments without cloud dependencies. You'll download models once, then query them through familiar OpenAI SDK calls — but everything runs offline on your hardware.
Why Ollama?

- Local-first: No API costs, no data sent to third parties
- OpenAI compatible: Use existing Python SDKs and agent frameworks
- Model management: Easy pull/list/ps commands for model lifecycle
- Performance optimized: GPU acceleration, streaming, context caching
- Developer friendly: REST API + CLI in one package

The great thing about Ollama being OpenAI compatible is that we can build code using OpenAI and locally hosted models, but then switch to high performing models like those hosted on tejas.   In most cases, we will only need to change our base_url and api_keys.  

## Ollama CLI interface 

Here are a few commonly used Ollama CLI commands to download, explore and use models. 
```bash 
# list models downloaded on your machine 
ollama list            

# download an additional model 
ollama pull qwen3     

# launches the local Ollama API/server, usually on port 11434. Can chat programattically with any model
ollama serve  
```

Additionally you can chat with models via command line.  Here are some additional useful commands for that 
```bash
# makes sure the model is available, then opens an interactive chat session with it through the server.
ollama run qwen3    

# checks which models are running 
ollama ps 

# stops model from running when you are no long using
ollama stop qwen3
```

We will focus on using Ollama programmatically in this tutorial. Let's first chat with an Ollama model with OpenAI. First we will use commands from above to make sure we have the correct models downloaded and local server started:

In [1]:
!ollama list 

]11;?\NAME                       ID              SIZE      MODIFIED     
qwen3:32b                  030ee887880f    20 GB     20 hours ago    
nomic-embed-text:latest    0a109f422b47    274 MB    4 days ago      
qwen3.5:latest             6488c96fa5fa    6.6 GB    11 days ago     
glm-5:cloud                c313cd065935    -         11 days ago     
kimi-k2.5:cloud            6d1c3246c608    -         11 days ago     
gpt-oss:20b                17052f91a42e    13 GB     5 weeks ago     
llama3.2:latest            a80c4f17acd5    2.0 GB    5 weeks ago     
qwen2.5:latest             845dbda0ea48    4.7 GB    6 months ago    
qwen3:4b                   359d7dd4bcda    2.5 GB    6 months ago    
qwen3-coder:30b            06c1097efce0    18 GB     6 months ago    
qwen3:1.7b                 8f68893c685c    1.4 GB    6 months ago    
qwen3:latest               500a1f067a9f    5.2 GB    8 months ago    
qwen2.5vl:32b              3edc3a52fe98    21 GB     8 months ago    
deepseek-r1:8b 

In [2]:
!ollama pull llama3.2:3b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [3]:
!ollama serve

]11;?\Error: listen tcp 127.0.0.1:11434: bind: address already in use


Now let's send a message the llamma3b with openai 

In [4]:
# Import OpenAI client class
from openai import OpenAI

In [5]:
# define variable for local model we will use 
llama3b = "llama3.2:3b"

# connect to ollama running in the backend 
openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# Define a wrapper function that will send requests to an LLM and receive responses
def generate_response(
        client: OpenAI,
        model: str,
        user_prompt: str,
        system_prompt: str = "You are a helpful assistant.",
        temperature: float = 0.5       
) -> str:
    """Sends a request to an LLM and and returns a response."""
    
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

# Vague prompt = vague result.
# We should expect a generic output from the LLM.
vague_prompt = "Tell me about Albert Einstein."

vague_response = generate_response(
    client=openai_client,
    model=llama3b,
    user_prompt=vague_prompt, 
)

print(vague_response)

Albert Einstein (1879-1955) was a renowned German-born physicist who is widely regarded as one of the most influential scientists of the 20th century. He is best known for his groundbreaking work on the theory of relativity, which revolutionized our understanding of space and time.

**Early Life and Education**

Einstein was born in Munich, Germany, to a Jewish family. His father was an engineer and salesman, and his mother was a homemaker. Einstein's early education was at a Catholic elementary school, followed by a year at a Protestant high school. He then enrolled in the Swiss Federal Polytechnic University, where he studied physics and mathematics.

**Career**

Einstein's academic career began with a master's degree in physics from the University of Zurich in 1900. He then worked as a patent clerk in Bern, Switzerland, while continuing his research on theoretical physics. In 1905, Einstein published four groundbreaking papers that transformed our understanding of space and time:

1

### Samba Nova Set Up

We can easily use the same function from above to chat with models available on tejas. 

FYI, you can find API docs for tejas here: https://ai.tejas.tacc.utexas.edu/

Let's use this to check which models are available with OpenAI.

In [31]:
# samba base url 
samba_base_url = "https://ai.tejas.tacc.utexas.edu"

# get api key from hidden file 
from dotenv import load_dotenv
import os 
load_dotenv()  # Loads .env into os.environ
samba_api_key = os.getenv("API_KEY")

In [32]:
# connect to tejas
openai_client_samba = OpenAI(
    base_url=samba_base_url,
    api_key=samba_api_key
)

In [34]:
# check what models are available 
models = openai_client_samba.models.list()
for model in models.data:
    print(f"ID: {model.id}, Owned by: {model.owned_by}")

ID: Llama-4-Maverick-17B-128E-Instruct, Owned by: openai
ID: Qwen3-32B, Owned by: openai
ID: Whisper-Large-v3, Owned by: openai
ID: E5-Mistral-7B-Instruct, Owned by: openai
ID: Meta-Llama-3.3-70B-Instruct, Owned by: openai
ID: Qwen3-32B-SC, Owned by: openai


Next we can send a message the tejas with the same functions generated for Ollama code. 

In [10]:
# define variable for local model we will use 
llama4 = "Llama-4-Maverick-17B-128E-Instruct"  # MOE model: 17B parameters used in inference and 128 Experts

# send a message to samba nova 
vague_response = generate_response(
    client=openai_client_samba,
    model=llama4,
    user_prompt=vague_prompt, 
)

print(vague_response)

Albert Einstein (1879-1955) was a renowned German-born physicist who is widely regarded as one of the most influential scientists of the 20th century. He is best known for his groundbreaking work on the theory of relativity and the famous equation E=mc².

**Early Life and Education**

Einstein was born in Munich, Germany, to Hermann and Pauline Einstein. He grew up in a middle-class Jewish family and was an average student in school. However, he was fascinated by science and mathematics, and spent much of his free time reading and thinking about these subjects. Einstein's curiosity and passion for learning were encouraged by his parents, who supported his interests.

Einstein studied physics at the Swiss Federal Polytechnic University, where he graduated in 1900. He then worked as a patent clerk in Bern, Switzerland, for seven years, during which time he completed his Ph.D. thesis and published several papers on theoretical physics.

**Theory of Relativity**

In 1905, Einstein's annus 

## Importing Academic Papers and Summarizing with LLMs 

In [12]:
!ls docs

manuscript_foundation_model_for_earth_system.pdf
supplement_foundation_model_for_earth_system.pdf


In [16]:
from pypdf import PdfReader

def read_pdf(path: str) -> str:
    reader = PdfReader(path)
    return "".join(page.extract_text() or "" for page in reader.pages)

# Read paper
pdf_text = read_pdf("docs/manuscript_foundation_model_for_earth_system.pdf")

# Inject into prompt (example)
prompt = f"""You are reviewing a research paper.
Here is the paper text:

{pdf_text}

Summarize the main contributions and methodology."""

summary = generate_response(
    client=openai_client_samba,
    model=llama4,
    user_prompt=prompt, 
)

print(summary)

The paper introduces Aurora, a large-scale foundation model for the Earth system, which is a machine learning model capable of producing forecasts for various Earth system variables at different resolutions. The main contributions and methodology are as follows:

**Main Contributions:**

1. **Aurora Model:** The authors propose a novel foundation model, Aurora, which is a flexible 3D Swin Transformer with 3D Perceiver-based atmospheric encoders and decoders.
2. **State-of-the-art Results:** Aurora achieves state-of-the-art performance in four critical forecasting domains: air quality, ocean waves, tropical cyclone tracks, and high-resolution weather forecasting.
3. **Efficient and Scalable:** Aurora is orders of magnitude faster than traditional numerical models and can be fine-tuned for diverse applications at modest expense.

**Methodology:**

1. **Pretraining:** Aurora is pretrained on a large body of Earth system data, including analysis, reanalysis, forecast, reforecast, and clima

## Extracting Meta Data with LLMs 

First PDFs do come with metadata that you can grab as seen in the simple code demo below:

In [17]:
# first PDFs do come with metadata that you can grab.  
reader = PdfReader("docs/manuscript_foundation_model_for_earth_system.pdf")
reader.metadata

{'/Author': 'Cristian Bodnar',
 '/CreationDate': "D:20250618124629+05'30'",
 '/Creator': 'Springer',
 '/CrossMarkDomains[1]': 'springer.com',
 '/CrossMarkDomains[2]': 'springerlink.com',
 '/Keywords': '',
 '/ModDate': "D:20250618124745+05'30'",
 '/Subject': 'Nature, doi:10.1038/s41586-025-09005-y',
 '/Title': 'A foundation model for the Earth system',
 '/doi': '10.1038/s41586-025-09005-y'}

However, there is likely paper specific data that you would like to obtain that is not included in the meta data above that may require reading the entire paper.  Thankfully, LLMs are pretty good and extracting this information particularly if you have a very specific factual inquiry.  Next, we will demo how we can get LLMs to extract this information from paper.

In [30]:
import instructor
from pydantic import BaseModel, Field
import json

instructor_client = instructor.from_openai(
    openai_client_samba,
    mode=instructor.Mode.JSON # When using with ollama, provide mode parameter 
)

# pydantic schema describing json data we would like returned
class MetaData(BaseModel):
    journal: str = Field(..., description="The journal the article was published in.")
    last_author: str = Field(..., description="The first and last name of the last author listed")

# prompt templates for extracting meta data
system_prompt = "You are a meta data extractor. Your task is to extract meta data from academic papers."
user_prompt = f"""Extract the meta data as specified in the data schema provided from the follwoing academic paper: 
                
                {pdf_text}
                """

# set this request to the sambanova
response = instructor_client.chat.completions.create(
    model=llama4,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_model=MetaData,
    max_retries=3,
)

# Convert Pydantic model to Python dictionary 
json_response = response.model_dump()

print(json_response)

{'journal': 'Nature', 'last_author': 'Paris Perdikaris'}
